In [ ]:
# %reload_ext autoreload
# %autoreload 2

In [ ]:
import sys
import os
import subprocess

IN_COLAB = False
try:
  from google.colab import drive
  IN_COLAB = True
  drive.mount('/content/drive')
except:
  pass

if IN_COLAB:
  REPO_PATH = "/content/MaskArchitectureAnomaly_CourseProject"

  if not os.path.exists(REPO_PATH):
    subprocess.run(
        ["git", "clone", "https://github.com/alberto467/MaskArchitectureAnomaly_CourseProject.git", REPO_PATH],
        check=True
    )

  CWD = f"{REPO_PATH}/eomt"
else:
  CWD = "./eomt"

sys.path.insert(0, CWD)
os.chdir(CWD)

DATA_PATH = "/content/drive/MyDrive/Project-FAIMDL/downloads" if IN_COLAB else "downloads"

Mounted at /content/drive


In [2]:
# In Colab, just install the packages missing from the base colab packages
!pip install -r requirements.colab.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 99.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.5/443.5 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Verify checkpoint path
CKPT_PATH = f"{DATA_PATH}/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt"
print("Checkpoint exists:", os.path.exists(CKPT_PATH))
print("Checkpoint path:", CKPT_PATH)

Checkpoint exists: True
Checkpoint path: /content/drive/MyDrive/POLITO/FAI/project/downloads/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt


# Model checkpoint review

In [35]:
# Inspect checkpoint content (trainer state vs weights-only)
import torch
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print("Checkpoint keys:", sorted(list(ckpt.keys()))[:20])
print("Has state_dict:", "state_dict" in ckpt)
print("Has optimizer_states:", "optimizer_states" in ckpt)
print("Has lr_schedulers:", "lr_schedulers" in ckpt)
print("Epoch:", ckpt.get("epoch"))
print("Global step:", ckpt.get("global_step"))

Checkpoint keys: ['MixedPrecision', 'callbacks', 'datamodule_hparams_name', 'datamodule_hyper_parameters', 'epoch', 'global_step', 'hparams_name', 'hyper_parameters', 'loops', 'lr_schedulers', 'optimizer_states', 'pytorch-lightning_version', 'state_dict']
Has state_dict: True
Has optimizer_states: True
Has lr_schedulers: True
Epoch: 23
Global step: 4461


In [ ]:
def scan_dict(d):
    for k, v in d.items():
        if isinstance(v, dict):
            scan_dict(v)
        if isinstance(v, torch.Tensor) and v.is_complex():
            print(k, v.shape, v.is_complex())

scan_dict(ckpt)

In [15]:
from copy import deepcopy

# Verify checkpoint path before resuming
src = f"{DATA_PATH}/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt"
# src = f"{DATA_PATH}/models/coco/eomt_coco.bin"
print("Checkpoint exists:", os.path.exists(src))
print("Checkpoint path:", src)
dst = src.replace(".ckpt", "_weights_only.bin")

# Load checkpoint
ckpt = torch.load(src, map_location="cpu")

print("Checkpoint keys:", sorted(list(ckpt.keys())))

if "state_dict" not in ckpt:
    raise ValueError("Checkpoint does not contain 'state_dict'. Using as-is...")

print("Checkpoint contains 'state_dict'. Extracting weights...")
ckpt = ckpt["state_dict"]
torch.save(ckpt, dst)


print(f"Saved cleaned checkpoint to:\n{dst}")

print("\nNew keys:")
print(sorted(ckpt.keys()))

Checkpoint exists: True
Checkpoint path: /content/drive/MyDrive/POLITO/FAI/project/downloads/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt
Checkpoint keys: ['MixedPrecision', 'callbacks', 'datamodule_hparams_name', 'datamodule_hyper_parameters', 'epoch', 'global_step', 'hparams_name', 'hyper_parameters', 'loops', 'lr_schedulers', 'optimizer_states', 'pytorch-lightning_version', 'state_dict']
Checkpoint contains 'state_dict'. Extracting weights...
Saved cleaned checkpoint to:
/content/drive/MyDrive/POLITO/FAI/project/downloads/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884_weights_only.bin

New keys:
['criterion.empty_weight', 'network.attn_mask_probs', 'network.class_head.bias', 'network.class_head.weight', 'network.encoder.backbone.blocks.0.attn.proj.bias', 'network.encoder.backbone.blocks.0.attn.proj.weight', 'network.encoder.backbone.blocks.0.attn.qkv.bias', 'network.encoder.backbone.blocks.0.attn.qkv.weight', 'network.encoder.backbone.blocks.0.ls1.gamma', 'network

# Training run

Currently setup for resuming from checkpoint

In [ ]:
# Run training with an explicit argument list (avoids shell formatting issues)

# We're using subprocess to avoid escape issues with weird paths (logs on weights and biases)
import subprocess
cmd = [
    "python3",
    "main.py",
    "fit",
    "-c",
    f"{CWD}/configs/dinov2/coco_city_finetune.yaml",
    "--data.path",
    f"{DATA_PATH}/datasets/cityscapes",
    # "--model.ckpt_path",
    # f"{DATA_PATH}/models/coco/eomt_coco.bin",
    "--ckpt_path",
    CKPT_PATH,
    # dst,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

Running: python3 main.py fit -c /content/eomt/configs/dinov2/coco_city_finetune.yaml --data.path /content/drive/MyDrive/POLITO/FAI/project/downloads/datasets/cityscapes --ckpt_path /content/drive/MyDrive/POLITO/FAI/project/downloads/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt


CalledProcessError: Command '['python3', 'main.py', 'fit', '-c', '/content/eomt/configs/dinov2/coco_city_finetune.yaml', '--data.path', '/content/drive/MyDrive/POLITO/FAI/project/downloads/datasets/cityscapes', '--ckpt_path', '/content/drive/MyDrive/POLITO/FAI/project/downloads/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt']' returned non-zero exit status 1.

# Validation run

In [37]:
# Run training with an explicit argument list (avoids shell formatting issues)
import subprocess
cmd = [
    "python3",
    "main.py",
    "validate",
    "-c",
    f"{CWD}/configs/dinov2/coco_city_finetune.yaml",
    "--data.path",
    f"{DATA_PATH}/datasets/cityscapes",
    "--ckpt_path",
    CKPT_PATH,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

Running: python3 main.py validate -c /content/eomt/configs/dinov2/coco_city_finetune.yaml --data.path /content/drive/MyDrive/POLITO/FAI/project/downloads/datasets/cityscapes --ckpt_path /content/drive/MyDrive/POLITO/FAI/project/downloads/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt


CompletedProcess(args=['python3', 'main.py', 'validate', '-c', '/content/eomt/configs/dinov2/coco_city_finetune.yaml', '--data.path', '/content/drive/MyDrive/POLITO/FAI/project/downloads/datasets/cityscapes', '--ckpt_path', '/content/drive/MyDrive/POLITO/FAI/project/downloads/checkpoints/best-epoch=23-metrics/val_iou_all=0.7884.ckpt'], returncode=0, stdout='Seed set to 0\nUsing 16bit Automatic Mixed Precision (AMP)\nGPU available: True (cuda), used: True\nTPU available: False, using: 0 TPU cores\nHPU available: False, using: 0 HPUs\n`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..\nwandb: Currently logged in as: alberto467 (alberto467-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin\nwandb: Tracking run with wandb version 0.19.10\nwandb: Run data is saved locally in /content/work/step5/wandb_logs/wandb/run-20260515_185326-9rb2flo7\nwandb: Run `wandb offline` to turn off syncing.\nwandb: Syncing run Coco Mo